In [4]:
import os
import csv
import datetime as dt
import pandas as pd

# index를 날짜로 하는 데이터프레임을 price, adj_price, divend에 대해 반환
def extract_stock_data(file_path, ticker):
    extract_datas = []

    with open(os.path.join(file_path, ticker + '.csv'), newline='') as csvfile:
        reader = csv.reader(csvfile)

        for row in reader:
            extract_datas.append(row)
        
    price_dates = list(map(dt.date.fromisoformat, extract_datas[0]))
    price_history = list(map(float, extract_datas[1]))
    adj_price_dates = list(map(dt.date.fromisoformat, extract_datas[2]))
    adj_price_history = list(map(float, extract_datas[3]))
    divend_dates = list(map(dt.date.fromisoformat, extract_datas[4]))
    divend_history = list(map(float, extract_datas[5]))

    price_df = pd.DataFrame({'date': price_dates, 'price': price_history})
    adj_price_df = pd.DataFrame({'date': adj_price_dates, 'adj_price': adj_price_history})
    divend_df = pd.DataFrame({'date': divend_dates, 'divend': divend_history})

    price_df.set_index('date', inplace=True)
    adj_price_df.set_index('date', inplace=True)
    divend_df.set_index('date', inplace=True)

    return (price_df, adj_price_df, divend_df)

def make_stock_data(file_path, tickers):
    stock_data = []

    for ticker in tickers:
        price_df, adj_price_df, divend_df = extract_stock_data(file_path, ticker)
        stock_df = pd.concat([price_df, adj_price_df, divend_df], axis=1, sort=True)
        stock_df.columns = pd.MultiIndex.from_product([[ticker], stock_df.columns])
        stock_data.append(stock_df)

    stock_data = pd.concat(stock_data, axis=1, sort=True)

    return stock_data


In [173]:
import itertools

class Portfolio:
    # stock_data
    def __init__(self, stock_data: pd.DataFrame, start_cash: float, target_ratio: dict, buy_ratio: float = 5.0, sell_ratio: float = 5.0):
        self.__stock_list = stock_data.columns.get_level_values(0).unique().to_list()
        self.__price_data = stock_data.loc[:, (stock_data.columns.get_level_values(1) == 'price')].copy()
        self.__price_data = self.__price_data.droplevel(1, axis=1)
        self.__adj_price_data = stock_data.loc[:, (stock_data.columns.get_level_values(1) == 'adj_price')].copy()
        self.__adj_price_data = self.__adj_price_data.droplevel(1, axis=1)
        self.__divend_data = stock_data.loc[:, (stock_data.columns.get_level_values(1) == 'divend')].copy()
        self.__divend_data = self.__divend_data.droplevel(1, axis=1)
        self.__start_cash = start_cash
        # target_ratio의 구성 { stock명 : 목표 비중, ... }
        self.target_ratio = target_ratio
        self.buy_ratio = buy_ratio
        self.sell_ratio = sell_ratio
        self.__target_buy_ratio = {ticker: target_ratio[ticker] * buy_ratio / 100 for ticker in self.stock_list}
        self.__target_sell_ratio = {ticker: target_ratio[ticker] * sell_ratio / 100 for ticker in self.stock_list}
    

        self.__price_data.dropna(inplace=True)
        self.__avilable_date = self.price_data.iloc[0].name
        self.__adj_price_data = self.__adj_price_data[self.__adj_price_data.index >= self.__avilable_date]
        self.__divend_data = self.__divend_data[self.__divend_data.index >= self.__avilable_date]
        
    @property
    def stock_list(self) -> list:
        return self.__stock_list
    @property
    def price_data(self) -> pd.DataFrame:
        return self.__price_data
    @property
    def adj_price_data(self) -> pd.DataFrame:
        return self.__adj_price_data
    @property
    def divend_data(self) -> pd.DataFrame:
        return self.__divend_data
    @property
    def start_date(self) -> dt.date:
        return self.__avilable_date

    def backtest(self, start_date=None):
        info_list =  ['price', 'number', 'value', 'weight']
        stock_info =  list(itertools.product(self.stock_list, info_list))

        col = [('total', 'value'), ('cash', 'value'), ('cash', 'weight')] + stock_info
        col = pd.MultiIndex.from_tuples(col)

        if not start_date or start_date < self.start_date:
            start_date = self.start_date
        dates = self.price_data[self.price_data.index >= start_date].index
        stat = pd.DataFrame(columns=col, index=dates)

        # 첫 값 설정
        first_date = dates[0]
        total_value = self.__start_cash
        for ticker in self.stock_list:
            stat.loc[first_date, ('total', 'value')] = total_value
            stat.loc[first_date, ('cash', 'value')] = 0
            stat.loc[first_date, ('cash', 'weight')] = 0
            stat.loc[:, (ticker, 'price')] = self.price_data[ticker]
            stat.loc[first_date, (stat.columns.get_level_values(0) == ticker) & (stat.columns.get_level_values(1) != 'price')] = self.__ideal_nvw(total_value, ticker, first_date)
            
        for i in range(1, len(dates)):
            cash_value = stat.loc[dates[i - 1]][('cash', 'value')]
            total_value = cash_value
            # 가격 변동 처리
            prev_num = stat.loc[dates[i - 1]][:, 'number']
            prev_num_pv = (prev_num * self.price_data.loc[dates[i]]).round(3)
            total_value += prev_num_pv.sum()
            # 배당금 처리
            divend_df = self.acculate_divend(dates[i-1], dates[i])
            divend = (divend_df.sum() * prev_num).sum().round(3)
            cash_value += divend
            total_value += divend
            stat.loc[dates[i], ('cash', 'value')] = cash_value
            stat.loc[dates[i], ('total', 'value')] = total_value

            # 변동에 따른 리밸런싱 계산
            prev_num_ratio = (prev_num_pv / total_value * 100).round(2)
            target_ratio_sr = pd.Series(self.target_ratio)
            ratio_diff = (prev_num_ratio - target_ratio_sr)

            need_sell = ratio_diff[ratio_diff >= pd.Series(self.__target_sell_ratio)].index.to_list()
            need_buy = ratio_diff[ratio_diff <= -pd.Series(self.__target_buy_ratio)].index.to_list()
            need_trade = need_sell + need_buy
            for ticker in self.stock_list:
                if ticker in need_trade:
                    stat.loc[dates[i], (stat.columns.get_level_values(0) == ticker) & (stat.columns.get_level_values(1) != 'price')] = self.__ideal_nvw(total_value, ticker, dates[i])
                    stat.loc[dates[i], ('cash', 'value')] += (ratio_diff[ticker] * total_value * 0.01).round(3)
                else:
                    stat.loc[dates[i], (ticker, 'number')] = stat.loc[dates[i - 1], (ticker, 'number')]
                    stat.loc[dates[i], (ticker, 'value')] = prev_num_pv[ticker]
                    stat.loc[dates[i], (ticker, 'weight')] = prev_num_ratio[ticker]
            
            stat.loc[dates[i], ('cash', 'weight')] = (stat.loc[dates[i]][('cash', 'value')] / total_value * 100).round(2)

        return stat

    # 범위는 (start_date, end_date]
    def acculate_divend(self, start_date, end_date):
        divend_df = self.divend_data[(self.divend_data.index > start_date) & (self.divend_data.index <= end_date)]
        return divend_df
    
    def stock_stdev(self, ticker=[], start_date=None, date_len=None, duration=1):
        if not ticker:
            ticker = self.stock_list

        if not start_date or start_date < self.start_date:
            start_date = self.start_date
        start_date = self.price_data[self.price_data.index >= start_date].index[0]
        start_idx = self.price_data.index.get_loc(start_date)
        
        max_len = len(self.price_data.index) - start_idx
        if not date_len:
            date_len = max_len
        date_len = min(date_len, max_len)
        end_idx = start_idx + date_len

        data_df = self.adj_price_data.loc[self.price_data.index[start_idx:end_idx], ticker]
        data_df = data_df.iloc[::duration]
        data_df['return'] = (((data_df.shift(-1)- data_df) / data_df[:-1]) * 100).round(2)
        data_df.dropna(inplace=True)

        # print(data_df)
        
        stdev = data_df['return'].std()

        print(stdev)
        return
    

    
    def __ideal_nvw(self, total_value, ticker, date):
        target_ratio = self.target_ratio[ticker] * 0.01
        value = round(total_value * target_ratio, 3)
        price = self.price_data.loc[date][ticker]
        number = round(value / price, 6)
        return (number, value, target_ratio * 100)

In [174]:
# stock 불러와서 데이터프레임화 하기
file_path = './etf'
tickers = ['QQQ', 'DGRW', 'SCHD']

stock_data = make_stock_data(file_path, tickers)

In [ ]:
p1 = Portfolio(stock_data, 10000, {'QQQ': 50, 'DGRW': 30, 'SCHD': 20}, start_date=dt.date(2017, 1, 1), buy_ratio=0, sell_ratio=0)
# p2 = Portfolio(stock_data.loc[:, ['QQQ']], 10000, {'QQQ': 100}, buy_ratio=0, sell_ratio=0)

# portfolio1 = Portfolio(stock_data.loc[:, 'QQQ'], 10000, {'QQQ': 100})
# stat = p2.backtest()
# p1.stock_stdev(ticker=['QQQ'], start_date=dt.date(2017, 1, 1), duration=4)
# p1.stock_stdev(ticker=['QQQ'], start_date=dt.date(2017, 1, 1), duration=52)
# p2.stock_stdev(ticker=['QQQ'], duration=5)
backtest_stat = p1.backtest()
print(backtest_stat)
# portfolio1 = portfolio.backtest()

5.480519943871081
26.815643651503954


In [168]:
backtest_stat.to_csv('./backtest_stat.csv')

NameError: name 'backtest_stat' is not defined